going to get looking at some data

In [ ]:
import pandas as pd
import numpy as np

big_data_filepath = "C:\\Users\\samib\\OneDrive - Imperial College London\\Imperial\\Education\\Year 4\\Masters Project\\Code\\MastersProject\\Data\\2015-2025jSPXoptions.csv"

big_data = pd.read_csv(big_data_filepath)



In [ ]:
# Inspect data structure
print("Dataset shape:", big_data.shape)
print("\nColumn names and types:")
print(big_data.info())
print("\nFirst few rows:")
print(big_data.head(5))
print("\nBasic statistics:")
print(big_data.describe())
print("\nData types:")
print(big_data.dtypes)
print("\nMissing values:")
print(big_data.isnull().sum())

In [ ]:
columns_to_keep = ["secid", "date", "exdate", "cp_flag", "strike_price", "best_bid", "best_offer",
                    "optionid", "exercise_style", "delta", "vega", "gamma", "impl_volatility"]

filtered_data = big_data[columns_to_keep]
# Rename strike_price to K and divide by 1000
filtered_data = filtered_data.rename(columns={'strike_price': 'K'})
filtered_data['K'] = filtered_data['K'] / 1000

print(filtered_data.head(5))

#about equal number of calls and puts
#only european options

In [ ]:
# Convert date column to datetime format (dd/mm/yyyy)
filtered_data['date'] = pd.to_datetime(filtered_data['date'], format='%d/%m/%Y')
filtered_data['exdate'] = pd.to_datetime(filtered_data['exdate'], format='%d/%m/%Y')

# Filter data to only include dates between 02/16/2016 and 01/07/2021
cutoff_date_start = pd.to_datetime('2016-10-16')
cutoff_date_end = pd.to_datetime('2021-07-01')
filtered_data = filtered_data[(filtered_data['date'] >= cutoff_date_start) & (filtered_data['date'] < cutoff_date_end)]

print(f"Dataset shape after date filter: {filtered_data.shape}")
print(f"\nDate range: {filtered_data['date'].min()} to {filtered_data['date'].max()}")
print(f"\nFirst few rows:")
print(filtered_data.head())

In [ ]:
# Load SPX price data
spx_filepath = "C:\\Users\\samib\\OneDrive - Imperial College London\\Imperial\\Education\\Year 4\\Masters Project\\Code\\MastersProject\\Data\\S&P 500 Historical Data 2015-2022.csv"  # Update filename if needed

spx_data = pd.read_csv(spx_filepath)

# Convert SPX date column to datetime format (mm/dd/yyyy)
spx_data['Date'] = pd.to_datetime(spx_data['Date'], format='%m/%d/%Y')

# Convert both date columns to date only (removing time component) for proper merging
# First ensure filtered_data['date'] is datetime before converting
if not pd.api.types.is_datetime64_any_dtype(filtered_data['date']):
    filtered_data['date'] = pd.to_datetime(filtered_data['date'])
    
filtered_data['date'] = filtered_data['date'].dt.date
filtered_data['exdate'] = filtered_data['exdate'].dt.date
spx_data['Date'] = spx_data['Date'].dt.date

print(spx_data.head())

# Merge filtered_data with spx_data on date columns
filtered_data = filtered_data.merge(spx_data[['Date', 'Price']], 
                                    left_on='date', 
                                    right_on='Date', 
                                    how='left')

# Rename 'Price' to 'S' and drop the duplicate 'Date' column
filtered_data = filtered_data.rename(columns={'Price': 'S'})
filtered_data = filtered_data.drop(columns=['Date'])



# Remove duplicate S columns, keeping only the first one
s_columns = [col for col in filtered_data.columns if col == 'S']
if len(s_columns) > 1:
    # Get all columns and remove duplicates
    filtered_data = filtered_data.loc[:, ~filtered_data.columns.duplicated(keep='first')]
    print("Removed duplicate columns")

print("Merge complete!")
print(f"Filtered data shape: {filtered_data.shape}")
print(f"\nFirst few rows:")
print(filtered_data.head())

In [ ]:
vix_filepath = r"C:\Users\samib\OneDrive - Imperial College London\Imperial\Education\Year 4\Masters Project\Code\MastersProject\Data\Full_VIX_History.csv"  # Update filename if needed

vix_data = pd.read_csv(vix_filepath)

# Convert VIX date column to datetime format (mm/dd/yyyy)
vix_data['DATE'] = pd.to_datetime(vix_data['DATE'], format='%m/%d/%Y')

# Convert DATE to date only (removing time component) to match filtered_data['date']
vix_data['DATE'] = vix_data['DATE'].dt.date


print(vix_data.head())

# Merge filtered_data with vix_data on date columns
filtered_data = filtered_data.merge(vix_data[['DATE', 'CLOSE']], 
                                    left_on='date', 
                                    right_on='DATE', 
                                    how='left')

# Rename 'CLOSE' to 'sigma' and drop the duplicate 'DATE' column
filtered_data = filtered_data.rename(columns={'CLOSE': 'sigma'})
filtered_data = filtered_data.drop(columns=['DATE'])

#turn sigma into percentage
filtered_data['sigma'] = filtered_data['sigma'] / 100

print("Merge complete!")
print(f"Filtered data shape: {filtered_data.shape}")
print(f"\nFirst few rows:")
print(filtered_data.head())

In [ ]:
#want to add interest rates now!!!
interestrate_filepath = "C:\\Users\\samib\\OneDrive - Imperial College London\\Imperial\\Education\\Year 4\\Masters Project\\Code\\MastersProject\\Data\\yield-curve-rates-2015-2025.csv"  # Update filename if needed

ir_data = pd.read_csv(interestrate_filepath)

def parse_mixed_date(date_str):
    try:
        # Try mm/dd/yyyy format first
        return pd.to_datetime(date_str, format='%m/%d/%Y')
    except:
        try:
            # Fall back to mm/dd/yy format
            return pd.to_datetime(date_str, format='%m/%d/%y')
        except:
            print(f"Unrecognized date format: {date_str}")

ir_data['Date'] = ir_data['Date'].apply(parse_mixed_date).dt.date

# Rename columns
ir_data = ir_data.rename(columns={
    '1 Mo': 'r 1 Mo',
    '3 Mo': 'r 3 Mo',
    '6 Mo': 'r 6 Mo'
})

# Merge interest rate data with filtered_data
filtered_data = filtered_data.merge(ir_data[['Date', 'r 1 Mo', 'r 3 Mo', 'r 6 Mo']], 
                                    left_on='date', 
                                    right_on='Date', 
                                    how='left')

# Drop the duplicate Date column
filtered_data = filtered_data.drop(columns=['Date'])

rate_cols = ['r 1 Mo', 'r 3 Mo', 'r 6 Mo']

# strip weird characters and coerce to numbers
for c in rate_cols:
    filtered_data[c] = (
        filtered_data[c]
        .astype(str)
        .str.replace('%','', regex=False)
        .str.replace(',','', regex=False)
        .str.strip()
    )
    filtered_data[c] = pd.to_numeric(filtered_data[c], errors='coerce')

# now divide by 100 (if the values are in percent units like 5.25)
filtered_data[rate_cols] = filtered_data[rate_cols] / 100


print("Interest rate merge complete!")
print(f"Filtered data shape: {filtered_data.shape}")
print(f"Columns: {list(filtered_data.columns)}")
print(f"\nFirst few rows:")
print(filtered_data.head())



In [ ]:
# Convert S and K to numeric types
filtered_data['S'] = pd.to_numeric(filtered_data['S'], errors='coerce')
filtered_data['K'] = pd.to_numeric(filtered_data['K'], errors='coerce')

# Convert date columns back to datetime for arithmetic operations
filtered_data['date'] = pd.to_datetime(filtered_data['date'])
filtered_data['exdate'] = pd.to_datetime(filtered_data['exdate'])

# Calculate new columns
# mid: midpoint of bid-ask spread
filtered_data['mid'] = (filtered_data['best_bid'] + filtered_data['best_offer']) / 2

# T: time to expiration in years
filtered_data['T'] = (filtered_data['exdate'] - filtered_data['date']).dt.days / 365.0

# Log-moneyness: log(S/K)
filtered_data['log_moneyness'] = np.log(filtered_data['S'] / filtered_data['K'])

# Remove duplicate S columns, keeping only the first one
s_columns = [col for col in filtered_data.columns if col == 'S']
if len(s_columns) > 1:
    # Get all columns and remove duplicates
    filtered_data = filtered_data.loc[:, ~filtered_data.columns.duplicated(keep='first')]
    print("Removed duplicate columns")

# Debug: Check cp_flag values before mapping
print("cp_flag values before mapping:")
print(filtered_data['cp_flag'].unique())
print(f"cp_flag data type: {filtered_data['cp_flag'].dtype}")
print(f"cp_flag null count: {filtered_data['cp_flag'].isnull().sum()}")

filtered_data["cp_flag"] = filtered_data["cp_flag"].map({"C": 1, "P": 0})

print("\nNew columns created!")
print(f"Filtered data shape: {filtered_data.shape}")
print(f"\nColumns: {list(filtered_data.columns)}")
print(f"\nFirst few rows:")
print(filtered_data.head())
print(filtered_data.tail())

# Print unique cp_flag values
print("\nUnique cp_flag values in training data:", filtered_data['cp_flag'].unique())
print("Unique cp_flag values in test data:", filtered_data['cp_flag'].unique())





In [ ]:
#putting in Black scholes price for feature
from math import erf, sqrt

def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def bs_price_row(S, K, T, r, sigma, cp_flag):
    # Basic guards
    if not np.isfinite(S) or not np.isfinite(K) or not np.isfinite(T) or not np.isfinite(r) or not np.isfinite(sigma):
        return np.nan
    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        return np.nan

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if cp_flag == 1:  # Call
        return S * norm_cdf(d1) - K * np.exp(-r * T) * norm_cdf(d2)
    else:            # Put
        return K * np.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)

# Choose your rate column
r_col = "r 3 Mo"

# Ensure rate is decimal (uncomment if your rates are in percent units)
# filtered_data[r_col] = filtered_data[r_col] / 100.0

# Compute BS price
filtered_data["bs_price"] = filtered_data.apply(
    lambda row: bs_price_row(row["S"], row["K"], row["T"], row[r_col], row["sigma"], row["cp_flag"]),
    axis=1
)

print("Added bs_price column.")
print(filtered_data[["S","K","T",r_col,"sigma","cp_flag","mid","bs_price", "delta", "vega", "gamma", "impl_volatility"]].head())

In [ ]:
# # Check if optionid 134606380 exists in the dataset
# target_id = 134606380
# if target_id in filtered_data['optionid'].values:
#     print(f"Found optionid {target_id}!")
#     print("\nRow(s) with this optionid:")
#     print(filtered_data[filtered_data['optionid'] == target_id])
# else:
#     print(f"optionid {target_id} NOT found in the dataset")

should probs build test data set before moving on


In [ ]:
option_filepath = "C:\\Users\\samib\\OneDrive - Imperial College London\\Imperial\\Education\\Year 4\\Masters Project\\Code\\MastersProject\\Data\\30day134606830.csv"

option = pd.read_csv(option_filepath)

# Convert dates to datetime and then to date for merging
option['date'] = pd.to_datetime(option['date'], format='%d/%m/%Y').dt.date
option['exdate'] = pd.to_datetime(option['exdate'], format='%d/%m/%Y')

# Rename strike_price to K and divide by 1000
option = option.rename(columns={'strike_price': 'K'})
option['K'] = option['K'] / 1000

option["cp_flag"] = "C"

# Merge with SPX data (Date column is already in date format from earlier processing)
option = option.merge(spx_data[['Date', 'Price']], 
                     left_on='date', right_on='Date', how='left')
option = option.rename(columns={'Price': 'S'}).drop(columns=['Date'])

# Merge with VIX data (DATE column is already in date format from earlier processing)
option = option.merge(vix_data[['DATE', 'CLOSE']], 
                     left_on='date', right_on='DATE', how='left')
option = option.rename(columns={'CLOSE': 'sigma'}).drop(columns=['DATE'])
#turn sigma into percentage
option['sigma'] = option['sigma'] / 100

# Merge with interest rate data (Date column is already in date format from earlier processing)
option = option.merge(ir_data[['Date', 'r 1 Mo', 'r 3 Mo', 'r 6 Mo']], 
                     left_on='date', right_on='Date', how='left')
option = option.drop(columns=['Date'])
rate_cols = ['r 1 Mo', 'r 3 Mo', 'r 6 Mo']

# strip weird characters and coerce to numbers
for c in rate_cols:
    option[c] = (
        option[c]
        .astype(str)
        .str.replace('%','', regex=False)
        .str.replace(',','', regex=False)
        .str.strip()
    )
    option[c] = pd.to_numeric(option[c], errors='coerce')
# now divide by 100 (if the values are in percent units like 5.25)
option[rate_cols] = option[rate_cols] / 100


# Convert to numeric and calculate derived columns
option['S'] = pd.to_numeric(option['S'], errors='coerce')
option['K'] = pd.to_numeric(option['K'], errors='coerce')
option['date'] = pd.to_datetime(option['date'])

option['mid'] = (option['best_bid'] + option['best_offer']) / 2
option['T'] = (option['exdate'] - option['date']).dt.days / 365.0
option['log_moneyness'] = np.log(option['S'] / option['K'])
option['moneyness'] = option['S'] / option['K']




columns = ["date","S", "K", "T", "log_moneyness", "moneyness", "sigma", "cp_flag", "mid", "r 1 Mo", "r 3 Mo", "r 6 Mo", "bs_price", "delta", "vega", "gamma", "impl_volatility"]

option["cp_flag"] = option["cp_flag"].map({"C": 1, "P": 0})
#going to put in black scholes price for feature
# Choose your rate column
r_col = "r 3 Mo"

# Ensure rate is decimal (uncomment if your rates are in percent units)
# filtered_data[r_col] = filtered_data[r_col] / 100.0

# Compute BS price
option["bs_price"] = option.apply(
    lambda row: bs_price_row(row["S"], row["K"], row["T"], row[r_col], row["sigma"], row["cp_flag"]),
    axis=1
)

print("Added bs_price column.")
print(option.head())


option_clean = option[columns]

print(option_clean.head())
print(option_clean.describe())



Going to start training the model on data, probs start small then increase it


In [ ]:
import numpy as np
import pandas as pd


from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error



TRAIN_FULL = filtered_data.copy(deep=True)
TEST_FULL  = option_clean.copy(deep=True)



In [ ]:
import numpy as np
import pandas as pd

FEATURES = ["T", "log_moneyness", "sigma", "r 3 Mo", "bs_price",
            "delta", "vega", "gamma", "impl_volatility"]

# ----------------------------
# 1) Rebuild base data
# ----------------------------
train_df = TRAIN_FULL.copy()
test_df  = TEST_FULL.copy()

train_df["date"] = pd.to_datetime(train_df["date"])
test_df["date"]  = pd.to_datetime(test_df["date"])

print("TRAIN_FULL shape:", train_df.shape)
print("TEST_FULL shape: ", test_df.shape)

# ----------------------------
# 2) Clean target + basic guards
# ----------------------------
train_df = train_df[np.isfinite(train_df["mid"]) & (train_df["mid"] > 0) & (train_df["T"] > 0)].copy()

# ----------------------------
# 3) Filters (time window + moneyness + maturity + calls only)
# ----------------------------
MIN_T, MAX_T = 0.0136986301, 1
MIN_M, MAX_M = -0.5, 0.5
cutoff_date_start = pd.to_datetime("2019-07-01")
cutoff_date_end   = pd.to_datetime("2021-06-30")

mask = (
    train_df["T"].between(MIN_T, MAX_T)
    & train_df["log_moneyness"].between(MIN_M, MAX_M)
    & (train_df["date"] >= cutoff_date_start)
    & (train_df["date"] <  cutoff_date_end)
)
train_df = train_df.loc[mask].copy()
train_df = train_df[train_df["cp_flag"] == 1].copy()

# ---- NEW: drop rows with NaN in any feature ----
n_before = len(train_df)
train_df = train_df.dropna(subset=FEATURES).copy()
n_after = len(train_df)
print(f"Dropped {n_before - n_after} rows with NaN in features "
      f"({100*(n_before - n_after)/n_before:.2f}%)")
# -------------------------------------------------

print(f"\nFiltered TRAIN date range: {train_df['date'].min()} to {train_df['date'].max()}")
print("Filtered TRAIN shape (calls only, no NaN):", train_df.shape)

# ----------------------------
# 4) Subsample (compute budget), then restore chronological order
# ----------------------------
SAMPLE_SIZE = min(1000000, len(train_df))
train_df = (train_df
            .sample(n=SAMPLE_SIZE, random_state=21)
            .sort_values("date")
            .reset_index(drop=True))
print("TRAIN shape after sampling + chronological sort:", train_df.shape)

# ----------------------------
# 5) Untouched test set (used ONLY for final evaluation)
# ----------------------------

test_df = test_df[np.isfinite(test_df["mid"]) & (test_df["mid"] > 0) & (test_df["T"] > 0)].copy()
test_df = test_df.dropna(subset=FEATURES).copy()  

X_test = test_df[FEATURES].copy()
y_test = test_df["mid"].copy()

print("\nUnique cp_flag in Train:", train_df["cp_flag"].unique())
print("Unique cp_flag in Test: ", test_df["cp_flag"].unique())

print("\nTraining features preview:")
print(train_df[FEATURES].head(10))







In [ ]:
import time


# xgboost = XGBRegressor(
#     n_estimators=50,
#     max_depth=100,
#     min_samples_leaf=1,
#     min_samples_split=10,
#     #max_features="sqrt",
#     #learning_rate=0.3,
#     random_state=42,

# )

# print("Starting model training...")
# start = time.time()

# xgboost.fit(X_train, y_train)

# elapsed = time.time() - start
# print(f"\nTraining completed in {elapsed/60:.2f} minutes ({elapsed:.1f} seconds)")




# pred = xgboost.predict(X_test)
# test_df["pred"] = pred

# mse = mean_squared_error(y_test, pred)
# rmse = np.sqrt(mse)
# mae = mean_absolute_error(y_test, pred)
# bias = float((pred - y_test).mean())

# print("\n=== Model Performance ===")
# print(f"MSE:  {mse:.6f}")
# print(f"RMSE: {rmse:.6f}")
# print(f"MAE:  {mae:.6f}")
# print(f"Bias: {bias:.6f}")

# train_pred = xgboost.predict(X_train)
# test_pred  = xgboost.predict(X_test)

# train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
# test_rmse  = np.sqrt(mean_squared_error(y_test, test_pred))

# print("Train RMSE:", train_rmse)
# print("Test RMSE:", test_rmse) #jsut a place holder need to validation stuff aftewars

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# Build chronologically sorted CV pool from the full filtered training set.
cv_df = train_df.sort_values("date").reset_index(drop=True)
X_cv  = cv_df[FEATURES].copy()
y_cv  = cv_df["mid"].copy()

tscv = TimeSeriesSplit(n_splits=5)

# ----------------------------
# Hyperparameter grid: aggressive XGBoost
# ----------------------------
N_ESTIMATORS_MAX  = 6000
EARLY_STOP_ROUNDS = 75
INNER_VAL_FRAC    = 0.15

list_learning_rate    = [0.02, 0.03, 0.05]
list_max_depth        = [5, 6, 7, 8]
list_min_child_weight = [1, 2]
list_reg_alpha        = [0.0, 0.01]
list_reg_lambda       = [0.1, 0.5, 1.0]
list_subsample        = [1.0]
list_colsample_bytree = [1.0]
list_of_hparams = [
    (lr, md, mcw, a, l, subsample, colsample)
    for lr in list_learning_rate
    for md in list_max_depth
    for mcw in list_min_child_weight
    for a in list_reg_alpha
    for l in list_reg_lambda
    for subsample in list_subsample
    for colsample in list_colsample_bytree
]

print("Total combos:", len(list_of_hparams))

best_val_rmse = np.inf
best_hparams  = None

results_df = pd.DataFrame(columns=[
    "method",
    "learning_rate",
    "max_depth",
    "min_child_weight",
    "reg_alpha",
    "reg_lambda",
    "subsample",
    "colsample_bytree",
    "rmse_train",
    "rmse_val",
    "overfit_gap",
    "best_iter_mean",
])

for i, (learning_rate, max_depth, min_child_weight, reg_alpha, reg_lambda, subsample, colsample_bytree) in enumerate(list_of_hparams):
    print(f"\nRun {i+1}/{len(list_of_hparams)}: "
          f"lr={learning_rate}, depth={max_depth}, mcw={min_child_weight}, "
          f"alpha={reg_alpha}, lambda={reg_lambda}, "
          f"subsample={subsample}, colsample={colsample_bytree}")

    fold_train_rmses, fold_val_rmses, fold_best_iters = [], [], []

    for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_cv)):
        X_tr_full, X_va = X_cv.iloc[tr_idx], X_cv.iloc[va_idx]
        y_tr_full, y_va = y_cv.iloc[tr_idx], y_cv.iloc[va_idx]

        # Chronological inner validation set for early stopping
        n_tr_full   = len(tr_idx)
        n_inner_val = max(1, int(INNER_VAL_FRAC * n_tr_full))
        n_inner_tr  = n_tr_full - n_inner_val

        X_inner_tr  = X_tr_full.iloc[:n_inner_tr]
        X_inner_val = X_tr_full.iloc[n_inner_tr:]
        y_inner_tr  = y_tr_full.iloc[:n_inner_tr]
        y_inner_val = y_tr_full.iloc[n_inner_tr:]

        xgboost = XGBRegressor(
            objective             = "reg:squarederror",
            learning_rate         = learning_rate,
            max_depth             = max_depth,
            min_child_weight      = min_child_weight,
            n_estimators          = N_ESTIMATORS_MAX,
            reg_alpha             = reg_alpha,
            reg_lambda            = reg_lambda,
            subsample             = subsample,
            colsample_bytree      = colsample_bytree,
            tree_method           = "hist",
            early_stopping_rounds = EARLY_STOP_ROUNDS,
            eval_metric           = "rmse",
            n_jobs                = -1,
            random_state          = 1,
        )

        xgboost.fit(
            X_inner_tr, y_inner_tr,
            eval_set=[(X_inner_val, y_inner_val)],
            verbose=False,
        )

        best_iter = int(xgboost.best_iteration) + 1

        fold_train_pred = xgboost.predict(
            X_tr_full,
            iteration_range=(0, best_iter)
        )

        fold_val_pred = xgboost.predict(
            X_va,
            iteration_range=(0, best_iter)
        )

        fold_train_rmse = np.sqrt(mean_squared_error(y_tr_full, fold_train_pred))
        fold_val_rmse   = np.sqrt(mean_squared_error(y_va, fold_val_pred))

        fold_train_rmses.append(fold_train_rmse)
        fold_val_rmses.append(fold_val_rmse)
        fold_best_iters.append(best_iter)

        print(f"   fold {fold_idx+1}/5 "
              f"| n_tr={n_inner_tr:>6} "
              f"| n_innerval={n_inner_val:>6} "
              f"| n_val={len(va_idx):>6} "
              f"| best_iter={best_iter:>4} "
              f"| train RMSE={fold_train_rmse:.4f} "
              f"| val RMSE={fold_val_rmse:.4f}")

    mean_train_rmse = float(np.mean(fold_train_rmses))
    mean_val_rmse   = float(np.mean(fold_val_rmses))
    mean_best_iter  = float(np.mean(fold_best_iters))
    overfit_gap     = mean_val_rmse - mean_train_rmse

    marker = "  <-- new best" if mean_val_rmse < best_val_rmse else ""

    print(f"   --> mean train RMSE={mean_train_rmse:.4f} "
          f"| mean val RMSE={mean_val_rmse:.4f} "
          f"| gap={overfit_gap:+.4f} "
          f"| mean best_iter={mean_best_iter:.0f}{marker}")

    if mean_val_rmse < best_val_rmse:
        best_val_rmse = mean_val_rmse
        best_hparams  = (
            learning_rate,
            max_depth,
            min_child_weight,
            reg_alpha,
            reg_lambda,
            subsample,
            colsample_bytree,
            mean_best_iter,
        )

    row = pd.DataFrame([[
        "XGBoost",
        learning_rate,
        max_depth,
        min_child_weight,
        reg_alpha,
        reg_lambda,
        subsample,
        colsample_bytree,
        mean_train_rmse,
        mean_val_rmse,
        overfit_gap,
        mean_best_iter,
    ]], columns=results_df.columns)

    results_df = pd.concat([results_df, row], ignore_index=True)

print("\nTop models by mean validation RMSE:")
print(results_df.sort_values("rmse_val").head(10))

print(f"\nBest hparams "
      f"(lr, depth, mcw, alpha, lambda, subsample, colsample, mean_best_iter): "
      f"{best_hparams}")

print(f"Best mean val RMSE = {best_val_rmse:.6f}")

In [ ]:
# -------------------------------------------------------
# FINAL MODEL: held-out tail + early stopping, then refit on full train
# -------------------------------------------------------
FINAL_LEARNING_RATE    = 0.05
FINAL_MAX_DEPTH        = 5
FINAL_REG_ALPHA        = 0.0
FINAL_REG_LAMBDA       = 0.5
FINAL_MIN_CHILD_WEIGHT = 2
N_ESTIMATORS_CAP       = 3000     # safety cap; early stopping picks the actual count
EARLY_STOP_ROUNDS      = 50
ES_TAIL_FRAC           = 0.15

import time
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

# --- Step 1: held-out chronological tail for early stopping ---
# train_df is already sorted by date in cell 16
n_total = len(train_df)
n_es    = max(1, int(ES_TAIL_FRAC * n_total))
n_inner = n_total - n_es

X_inner = train_df[FEATURES].iloc[:n_inner]
y_inner = train_df["mid"].iloc[:n_inner]
X_es    = train_df[FEATURES].iloc[n_inner:]
y_es    = train_df["mid"].iloc[n_inner:]

print(f"Step 1: fitting on {n_inner} rows, early-stop tail = {n_es} rows")

xgb_es = XGBRegressor(
    learning_rate         = FINAL_LEARNING_RATE,
    max_depth             = FINAL_MAX_DEPTH,
    n_estimators          = N_ESTIMATORS_CAP,
    reg_alpha             = FINAL_REG_ALPHA,
    reg_lambda            = FINAL_REG_LAMBDA,
    min_child_weight      = FINAL_MIN_CHILD_WEIGHT,
    objective             = "reg:squarederror",
    eval_metric           = "rmse",
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    tree_method           = "hist",
    early_stopping_rounds = EARLY_STOP_ROUNDS,
    n_jobs                = -1,
    random_state          = 42,
)

start = time.time()
xgb_es.fit(X_inner, y_inner, eval_set=[(X_es, y_es)], verbose=False)
best_iter = int(xgb_es.best_iteration) + 1
print(f"Step 1 done in {(time.time()-start)/60:.2f} min; best_iter = {best_iter}")

# --- Step 2: refit on full train_df with the chosen iteration count ---
print(f"Step 2: refitting on full {n_total} rows with n_estimators = {best_iter}")

X_full_train = train_df[FEATURES].copy()
y_full_train = train_df["mid"].copy()

xgb_final = XGBRegressor(
    learning_rate    = FINAL_LEARNING_RATE,
    max_depth        = FINAL_MAX_DEPTH,
    n_estimators     = best_iter,
    reg_alpha        = FINAL_REG_ALPHA,
    reg_lambda       = FINAL_REG_LAMBDA,
    min_child_weight = FINAL_MIN_CHILD_WEIGHT,
    objective        = "reg:squarederror",
    eval_metric      = "rmse",
    subsample        = 0.8,
    colsample_bytree = 0.8,
    tree_method      = "hist",
    n_jobs           = -1,
    random_state     = 42,
)

start = time.time()
xgb_final.fit(X_full_train, y_full_train)
print(f"Step 2 done in {(time.time()-start)/60:.2f} min")

# --- Evaluate ---
pred_train_final = xgb_final.predict(X_full_train)
pred_test_final  = xgb_final.predict(X_test)
pred_test        = pred_test_final   # so cell 21 still works

def report(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    bias = float((y_pred - y_true).mean())
    print(f"\n=== {name} ===")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  MAE:  {mae:.6f}")
    print(f"  Bias: {bias:.6f}")
    return rmse, mae, bias

rmse_tr, _, _ = report("TRAIN (full)", y_full_train, pred_train_final)
rmse_te, _, _ = report("TEST",         y_test,       pred_test_final)
print(f"\nGeneralisation gap (Test - Train RMSE): {rmse_te - rmse_tr:+.6f}")















# import time
# # --- Chosen model ---
# xgboost = XGBRegressor(
#     n_estimators=3000,
#     learning_rate=0.01,
#     max_depth=7,

#     reg_alpha=0,
#     reg_lambda=1.5,
#     gamma=0.0,
#     objective="reg:squarederror",
#     eval_metric="rmse"
# )



# def eval_split(model, X, y, name=""):
#     pred = model.predict(X)
#     mse  = mean_squared_error(y, pred)
#     rmse = np.sqrt(mse)
#     mae  = mean_absolute_error(y, pred)
#     bias = float((pred - y).mean())
#     print(f"\n=== {name} Performance ===")
#     print(f"MSE:  {mse:.6f}")
#     print(f"RMSE: {rmse:.6f}")
#     print(f"MAE:  {mae:.6f}")
#     print(f"Bias: {bias:.6f}")
#     return pred, rmse, mae, bias

# print("Starting model training...")
# start = time.time()

# xgboost.fit(X_train, y_train)

# elapsed = time.time() - start
# print(f"\nTraining completed in {elapsed/60:.2f} minutes ({elapsed:.1f} seconds)")

# # --- Evaluate on Train / Val / Test ---
# pred_train, rmse_train, mae_train, bias_train = eval_split(xgboost, X_train, y_train, "TRAIN")
# pred_val,   rmse_val,   mae_val,   bias_val   = eval_split(xgboost, X_val,   y_val,   "VALIDATION")
# pred_test,  rmse_test,  mae_test,  bias_test  = eval_split(xgboost, X_test,  y_test,  "TEST")

# print("\n=== Generalisation Gaps ===")
# print(f"Val - Train RMSE gap:  {rmse_val - rmse_train:.6f}")
# print(f"Test - Train RMSE gap: {rmse_test - rmse_train:.6f}")





















# with open('best_XG_val.pkl','rb') as f:
#     xg_best = pickle.load(f)

# xg_best.get_params()
# xg_best.fit(X_train, y_train)


# pred = xg_best.predict(X_test)
# test_df["pred"] = pred

# mse = mean_squared_error(y_test, pred)
# rmse = np.sqrt(mse)
# mae = mean_absolute_error(y_test, pred)
# bias = float((pred - y_test).mean())

# print("\n=== Model Performance ===")
# print(f"MSE:  {mse:.6f}")
# print(f"RMSE: {rmse:.6f}")
# print(f"MAE:  {mae:.6f}")
# print(f"Bias: {bias:.6f}")


# train_pred = xg_best.predict(X_train)
# pred_test  = xg_best.predict(X_test)

# train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
# test_rmse  = np.sqrt(mean_squared_error(y_test, pred_test))

# print("Train RMSE:", train_rmse)
# print("Test RMSE:", test_rmse)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

print(pred_test)

fig, ax = plt.subplots(figsize=(12,6))

ax.plot(test_df["date"], test_df["mid"], 
        label="Market Mid", linewidth=2)

ax.plot(test_df["date"], pred_test, 
        label="RF Prediction", linewidth=2, linestyle="--")

ax.set_title("Option Price: Market vs Random Forest Prediction", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Option Price")

ax.legend()
ax.grid(True, alpha=0.3)

# Better date formatting
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()
